# Exploratory Data Analysis
## ML-Based Product Recommendation System for Young Adult Customers on Daraz Nepal

**Author:** Binnol Dahal | **Coventry ID:** 14809734  
**Module:** ST6000CEM Individual Project Preparation  
**Module Leader:** Manoj Shrestha

---
This notebook performs a comprehensive EDA of the synthetic Daraz Nepal dataset.  
It covers user demographics, product distributions, interaction patterns,
festival effects, RFM segmentation, and feature correlations.


## 0. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Style ──────────────────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor':   '#FAFAFA',
    'axes.grid':        True,
    'grid.alpha':       0.3,
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.family':      'sans-serif',
    'axes.titlesize':   13,
    'axes.labelsize':   11,
})
ORANGE = '#F85606'
BLUE   = '#2979FF'
GREEN  = '#00C853'
GOLD   = '#FFB300'
PURPLE = '#9C27B0'
COLORS = [ORANGE, BLUE, GREEN, GOLD, PURPLE, '#00BCD4']

# ── Load data ──────────────────────────────────────────────────────────────
import os
BASE = os.path.join(os.getcwd(), '..', 'data')

users        = pd.read_csv(f'{BASE}/users.csv')
products     = pd.read_csv(f'{BASE}/products.csv')
interactions = pd.read_csv(f'{BASE}/cleaned_interactions.csv', parse_dates=['timestamp'])
user_feat    = pd.read_csv(f'{BASE}/user_features.csv')
prod_feat    = pd.read_csv(f'{BASE}/product_features.csv')

interactions['festival_context'] = interactions['festival_context'].fillna('None').astype(str)
interactions.loc[interactions['festival_context']=='nan','festival_context'] = 'None'

print(f"Users        : {len(users):,}")
print(f"Products     : {len(products):,}")
print(f"Interactions : {len(interactions):,}")
print(f"Date range   : {interactions['timestamp'].min().date()} → {interactions['timestamp'].max().date()}")

---
## 1. User Demographics

Understanding who our users are is the foundation of any personalisation system.
All users are young adults (18–30) reflecting Daraz Nepal's core customer base.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('User Demographics — Daraz Nepal (N=10,000)', fontsize=15, fontweight='bold', y=1.01)

# Age distribution
ax = axes[0,0]
ax.hist(users['age'], bins=13, color=ORANGE, edgecolor='white', linewidth=0.8)
ax.axvline(users['age'].mean(), color='black', linestyle='--', linewidth=1.5,
           label=f"Mean = {users['age'].mean():.1f}")
ax.set_title('Age Distribution'); ax.set_xlabel('Age'); ax.set_ylabel('Count')
ax.legend(fontsize=10)

# Gender
ax = axes[0,1]
gender_counts = users['gender'].value_counts()
wedges, texts, autotexts = ax.pie(gender_counts.values,
    labels=gender_counts.index, autopct='%1.1f%%',
    colors=[BLUE, ORANGE, GREEN], startangle=90,
    wedgeprops={'edgecolor':'white','linewidth':2})
ax.set_title('Gender Distribution')

# City (top 8)
ax = axes[0,2]
city_top = users['city'].value_counts().head(8)
bars = ax.barh(city_top.index[::-1], city_top.values[::-1], color=BLUE, edgecolor='white')
bars[0].set_color(ORANGE)
ax.set_title('Top Cities'); ax.set_xlabel('Users')

# Income level
ax = axes[1,0]
income_order = ['Low','Medium','High']
income_counts = users['income_level'].value_counts().reindex(income_order)
bars = ax.bar(income_counts.index, income_counts.values,
              color=[GREEN, BLUE, GOLD], edgecolor='white', linewidth=0.8)
for bar, val in zip(bars, income_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+50,
            f'{val:,}', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Income Level Distribution'); ax.set_ylabel('Users')

# Device type
ax = axes[1,1]
device_counts = users['device_type'].value_counts()
ax.bar(device_counts.index, device_counts.values,
       color=[ORANGE, BLUE, GREEN][:len(device_counts)], edgecolor='white')
ax.set_title('Device Type'); ax.set_ylabel('Users')

# Preferred category
ax = axes[1,2]
cat_counts = users['preferred_category'].value_counts()
ax.barh(cat_counts.index[::-1], cat_counts.values[::-1],
        color=COLORS[:len(cat_counts)], edgecolor='white')
ax.set_title('Preferred Category'); ax.set_xlabel('Users')

plt.tight_layout()
plt.savefig('user_demographics.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Remittance receivers: {users['remittance_receiver'].sum():,} "
      f"({users['remittance_receiver'].mean():.1%})")

---
## 2. Product Catalog Analysis

The catalog spans 6 categories with 1,000 products priced in Nepali Rupees (NPR).
Price distributions are right-skewed (log-normal) reflecting real e-commerce markets.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Product Catalog Analysis (N=1,000)', fontsize=15, fontweight='bold')

# Category distribution
ax = axes[0,0]
cat_counts = products['category'].value_counts()
ax.bar(cat_counts.index, cat_counts.values, color=COLORS, edgecolor='white')
ax.set_xticklabels(cat_counts.index, rotation=30, ha='right', fontsize=9)
ax.set_title('Products per Category'); ax.set_ylabel('Count')

# Price distribution (log scale)
ax = axes[0,1]
ax.hist(products['price_npr'], bins=40, color=BLUE, edgecolor='white', linewidth=0.5)
ax.set_xscale('log')
ax.axvline(products['price_npr'].median(), color=ORANGE, linestyle='--',
           linewidth=1.5, label=f"Median = NPR {products['price_npr'].median():,.0f}")
ax.set_title('Price Distribution (log scale)')
ax.set_xlabel('Price (NPR)'); ax.legend(fontsize=9)

# Price by category
ax = axes[0,2]
cat_order = products.groupby('category')['price_npr'].median().sort_values().index
bp = ax.boxplot([products[products['category']==c]['price_npr'].values
                 for c in cat_order],
                labels=[c.replace(' & ',' &\n') for c in cat_order],
                patch_artist=True, notch=False)
for patch, color in zip(bp['boxes'], COLORS):
    patch.set_facecolor(color); patch.set_alpha(0.7)
ax.set_yscale('log'); ax.set_title('Price Range by Category')
ax.set_ylabel('Price NPR (log)'); ax.tick_params(axis='x', labelsize=8)

# Rating distribution
ax = axes[1,0]
ax.hist(products['rating'], bins=20, color=GOLD, edgecolor='white')
ax.axvline(products['rating'].mean(), color='black', linestyle='--',
           label=f"Mean = {products['rating'].mean():.2f}")
ax.set_title('Product Rating Distribution')
ax.set_xlabel('Rating (1–5)'); ax.legend(fontsize=9)

# Review count (log)
ax = axes[1,1]
ax.hist(products['review_count'], bins=40, color=GREEN, edgecolor='white', linewidth=0.5)
ax.set_xscale('log')
ax.set_title('Review Count Distribution (log)')
ax.set_xlabel('Review Count')

# Festival relevance
ax = axes[1,2]
fest_counts = products['festival_relevance'].value_counts()
colors_f = [ORANGE if f != 'None' else '#CCC' for f in fest_counts.index]
ax.bar(fest_counts.index, fest_counts.values, color=colors_f, edgecolor='white')
ax.set_title('Festival Relevance Labels'); ax.set_ylabel('Products')

plt.tight_layout()
plt.savefig('product_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nPrice summary (NPR):")
print(products.groupby('category')['price_npr'].describe()[['min','50%','max']].round(0))

---
## 3. Interaction Patterns

Interactions follow a realistic funnel: views → wishlists → purchases.
The interaction weight scheme (view=1, wishlist=2, purchase=5) reflects
the relative value of each signal for collaborative filtering.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Interaction Patterns (N=446,028)', fontsize=15, fontweight='bold')

# Funnel
ax = axes[0,0]
int_counts = interactions['interaction_type'].value_counts()
total = len(interactions)
colors_f = [BLUE, GOLD, ORANGE]
bars = ax.bar(int_counts.index, int_counts.values, color=colors_f, edgecolor='white')
for bar, val in zip(bars, int_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
            f'{val:,}\n({val/total:.1%})', ha='center', fontsize=10, fontweight='bold')
ax.set_title('Interaction Funnel'); ax.set_ylabel('Count')

# Monthly trend
ax = axes[0,1]
monthly = interactions.groupby('month').size()
bar_colors = [ORANGE if m in [10,11] else BLUE for m in monthly.index]
bars = ax.bar(monthly.index, monthly.values, color=bar_colors, edgecolor='white')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_title('Monthly Interaction Volume')
ax.set_xlabel('Month'); ax.set_ylabel('Interactions')
orange_patch = mpatches.Patch(color=ORANGE, label='Festival months (Oct/Nov)')
blue_patch   = mpatches.Patch(color=BLUE,   label='Regular months')
ax.legend(handles=[orange_patch, blue_patch], fontsize=9)

# Hourly pattern
ax = axes[0,2]
hourly = interactions.groupby('hour').size()
ax.plot(hourly.index, hourly.values, color=ORANGE, linewidth=2.5, marker='o', markersize=4)
ax.fill_between(hourly.index, hourly.values, alpha=0.15, color=ORANGE)
ax.set_title('Hourly Shopping Pattern')
ax.set_xlabel('Hour of Day (0–23)'); ax.set_ylabel('Interactions')
peak_hour = hourly.idxmax()
ax.axvline(peak_hour, color='black', linestyle='--', linewidth=1,
           label=f'Peak: {peak_hour}:00')
ax.legend(fontsize=9)

# Day of week
ax = axes[1,0]
dow = interactions.groupby('dayofweek').size()
days = ['Mon','Tue','Wed','Thu','Fri','Sat','Sun']
bar_colors_d = [GREEN if d >= 5 else BLUE for d in dow.index]
ax.bar([days[d] for d in dow.index], dow.values, color=bar_colors_d, edgecolor='white')
ax.set_title('Day of Week Pattern'); ax.set_ylabel('Interactions')

# Interactions per user (distribution)
ax = axes[1,1]
user_int_counts = interactions.groupby('user_id').size()
ax.hist(user_int_counts.values, bins=50, color=PURPLE, edgecolor='white', linewidth=0.5)
ax.set_xscale('log')
ax.axvline(user_int_counts.median(), color=ORANGE, linestyle='--',
           label=f'Median = {user_int_counts.median():.0f}')
ax.set_title('Interactions per User (log)'); ax.legend(fontsize=9)

# Festival vs non-festival
ax = axes[1,2]
fest_group = interactions.groupby(['festival_context','interaction_type']).size().unstack(fill_value=0)
fest_group = fest_group.reindex(['None','Dashain','Tihar'])
fest_group.plot(kind='bar', ax=ax, color=[BLUE,GOLD,ORANGE], edgecolor='white', rot=0)
ax.set_title('Interactions by Festival Context')
ax.set_xlabel('Festival'); ax.set_ylabel('Count')
ax.legend(title='Type', fontsize=9)

plt.tight_layout()
plt.savefig('interaction_patterns.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFestival uplift:")
fest = interactions[interactions['festival_context']!='None']
non  = interactions[interactions['festival_context']=='None']
print(f"  Festival avg interactions/day  : {len(fest)/30:.0f}")
print(f"  Non-festival avg interactions/day: {len(non)/335:.0f}")
print(f"  Uplift: {len(fest)/30 / (len(non)/335):.2f}x")

---
## 4. RFM Segmentation

RFM (Recency, Frequency, Monetary) is a well-established customer segmentation
technique. Users are scored 1–5 on each dimension and grouped into segments
that drive adaptive recommendation weighting.


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('RFM Customer Segmentation', fontsize=15, fontweight='bold')

seg_colors = {
    'Champions': '#856404', 'Loyal': '#065F46',
    'Potential': '#1E40AF', 'At Risk': '#9F1239', 'Lost': '#6B7280'
}
seg_bg = {
    'Champions': '#FFF3CD', 'Loyal': '#D1FAE5',
    'Potential': '#DBEAFE', 'At Risk': '#FFE4E6', 'Lost': '#F3F4F6'
}

# Segment distribution
ax = axes[0,0]
if 'segment' in user_feat.columns:
    seg_counts = user_feat['segment'].value_counts()
    colors_s   = [seg_bg.get(s,'#EEE') for s in seg_counts.index]
    edge_s     = [seg_colors.get(s,'#999') for s in seg_counts.index]
    bars = ax.bar(seg_counts.index, seg_counts.values, color=colors_s,
                  edgecolor=edge_s, linewidth=2)
    for bar, val in zip(bars, seg_counts.values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10,
                str(val), ha='center', fontsize=10, fontweight='bold')
    ax.set_title('RFM Segment Distribution'); ax.set_ylabel('Users')
    ax.set_xticklabels(seg_counts.index, rotation=15)

# Recency distribution
ax = axes[0,1]
if 'recency' in user_feat.columns:
    ax.hist(user_feat['recency'].clip(upper=365), bins=40,
            color=BLUE, edgecolor='white', linewidth=0.5)
    ax.axvline(user_feat['recency'].median(), color=ORANGE, linestyle='--',
               label=f"Median = {user_feat['recency'].median():.0f} days")
    ax.set_title('Recency Distribution'); ax.set_xlabel('Days since last purchase')
    ax.legend(fontsize=9)

# Frequency distribution
ax = axes[0,2]
if 'frequency' in user_feat.columns:
    freq_clip = user_feat['frequency'].clip(upper=100)
    ax.hist(freq_clip, bins=40, color=GREEN, edgecolor='white', linewidth=0.5)
    ax.axvline(user_feat['frequency'].median(), color=ORANGE, linestyle='--',
               label=f"Median = {user_feat['frequency'].median():.0f} purchases")
    ax.set_title('Frequency Distribution'); ax.set_xlabel('Number of Purchases')
    ax.legend(fontsize=9)

# Monetary distribution (log)
ax = axes[1,0]
if 'monetary' in user_feat.columns:
    mon_pos = user_feat[user_feat['monetary']>0]['monetary']
    ax.hist(mon_pos, bins=40, color=GOLD, edgecolor='white', linewidth=0.5)
    ax.set_xscale('log')
    ax.axvline(mon_pos.median(), color=ORANGE, linestyle='--',
               label=f"Median = NPR {mon_pos.median():,.0f}")
    ax.set_title('Monetary Value Distribution (log)')
    ax.set_xlabel('Total Spend (NPR)'); ax.legend(fontsize=9)

# RFM score by segment
ax = axes[1,1]
if all(c in user_feat.columns for c in ['segment','RFM_score']):
    seg_order = ['Champions','Loyal','Potential','At Risk','Lost']
    seg_order = [s for s in seg_order if s in user_feat['segment'].unique()]
    rfm_by_seg = [user_feat[user_feat['segment']==s]['RFM_score'].values
                  for s in seg_order]
    bp = ax.boxplot(rfm_by_seg, labels=seg_order, patch_artist=True)
    for patch, seg in zip(bp['boxes'], seg_order):
        patch.set_facecolor(seg_bg.get(seg,'#EEE'))
        patch.set_edgecolor(seg_colors.get(seg,'#999'))
        patch.set_linewidth(2)
    ax.set_title('RFM Score by Segment'); ax.set_ylabel('RFM Score')
    ax.set_xticklabels(seg_order, rotation=15)

# Freq vs Monetary scatter
ax = axes[1,2]
if all(c in user_feat.columns for c in ['frequency','monetary','segment']):
    for seg in user_feat['segment'].unique():
        sub = user_feat[user_feat['segment']==seg].sample(min(100,len(user_feat)))
        ax.scatter(sub['frequency'], sub['monetary'],
                   color=seg_bg.get(seg,'#EEE'),
                   edgecolors=seg_colors.get(seg,'#999'),
                   alpha=0.7, s=30, label=seg, linewidths=1)
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_title('Frequency vs Monetary (log-log)')
    ax.set_xlabel('Frequency'); ax.set_ylabel('Monetary (NPR)')
    ax.legend(fontsize=8, markerscale=1.5)

plt.tight_layout()
plt.savefig('rfm_segmentation.png', dpi=150, bbox_inches='tight')
plt.show()

if 'segment' in user_feat.columns:
    print("\nRFM Segment Summary:")
    print(user_feat.groupby('segment')[['frequency','monetary','RFM_score']]
          .mean().round(2))

---
## 5. Nepal Festival Shopping Analysis

A key research contribution is modelling Nepal-specific festival behaviour.
Dashain (Oct) and Tihar (Nov) drive significant spikes in electronics and fashion
respectively, reflecting cultural gift-giving traditions.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Nepal Festival Shopping Patterns', fontsize=15, fontweight='bold')

# Electronics purchases by month
ax = axes[0]
elec_products = products[products['category']=='Electronics']['product_id']
elec_ints     = interactions[interactions['product_id'].isin(elec_products) &
                              (interactions['interaction_type']=='purchase')]
elec_monthly  = elec_ints.groupby('month').size()
colors_e      = [ORANGE if m in [10,11] else BLUE for m in elec_monthly.index]
ax.bar(elec_monthly.index, elec_monthly.values, color=colors_e, edgecolor='white')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_title('Electronics Purchases by Month')
ax.set_xlabel('Month'); ax.set_ylabel('Purchases')
ax.annotate('Dashain\nspike', xy=(10, elec_monthly[10]),
            xytext=(8, elec_monthly[10]+30),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

# Fashion purchases by month
ax = axes[1]
fash_products = products[products['category']=='Fashion']['product_id']
fash_ints     = interactions[interactions['product_id'].isin(fash_products) &
                              (interactions['interaction_type']=='purchase')]
fash_monthly  = fash_ints.groupby('month').size()
colors_f2     = [GOLD if m == 11 else '#FFB30055' if m==10 else '#CCC'
                 for m in fash_monthly.index]
ax.bar(fash_monthly.index, fash_monthly.values, color=colors_f2, edgecolor='white')
ax.set_xticks(range(1,13))
ax.set_xticklabels(['J','F','M','A','M','J','J','A','S','O','N','D'])
ax.set_title('Fashion Purchases by Month')
ax.set_xlabel('Month')
ax.annotate('Tihar\nspike', xy=(11, fash_monthly[11]),
            xytext=(9, fash_monthly[11]+20),
            arrowprops=dict(arrowstyle='->', color='black'), fontsize=9)

# Festival uplift by category
ax = axes[2]
categories = products['category'].unique()
uplifts = []
for cat in categories:
    cat_p   = products[products['category']==cat]['product_id']
    cat_i   = interactions[interactions['product_id'].isin(cat_p)]
    fest    = cat_i[cat_i['festival_context']!='None']
    nonfest = cat_i[cat_i['festival_context']=='None']
    n_fest_days = 18
    n_non_days  = 347
    avg_fest    = len(fest)  / n_fest_days
    avg_non     = len(nonfest) / n_non_days
    uplift      = avg_fest / max(avg_non, 1)
    uplifts.append({'category': cat, 'uplift': uplift})

uplift_df = pd.DataFrame(uplifts).sort_values('uplift', ascending=True)
colors_u  = [ORANGE if u > 1.5 else BLUE for u in uplift_df['uplift']]
ax.barh(uplift_df['category'], uplift_df['uplift'], color=colors_u, edgecolor='white')
ax.axvline(1.0, color='black', linestyle='--', linewidth=1, label='No uplift (1.0x)')
ax.set_title('Festival Uplift by Category')
ax.set_xlabel('Uplift multiplier (festival vs non-festival)')
ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('festival_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print("Festival interactions breakdown:")
print(interactions.groupby('festival_context')['interaction_type']
      .value_counts().unstack(fill_value=0))

---
## 6. User-Item Matrix Sparsity Analysis

High matrix sparsity is the fundamental challenge that motivates hybrid modelling.
A 95.7% sparse matrix means collaborative filtering alone struggles — content-based
signals and demographic features become essential supplements.


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('User-Item Matrix Sparsity Analysis', fontsize=15, fontweight='bold')

# Build small sample matrix for visualisation
weight_map = {'view':1,'wishlist':2,'purchase':5}
interactions['w'] = interactions['interaction_type'].map(weight_map)

sample_users = interactions['user_id'].value_counts().head(80).index
sample_items = interactions['product_id'].value_counts().head(80).index

sub = interactions[interactions['user_id'].isin(sample_users) &
                   interactions['product_id'].isin(sample_items)]
mat = sub.groupby(['user_id','product_id'])['w'].max().unstack(fill_value=0)

# Heatmap sample
ax = axes[0]
sns.heatmap(mat.values, ax=ax, cmap='Oranges', cbar_kws={'label':'Weight'},
            xticklabels=False, yticklabels=False)
ax.set_title('Sample 80×80 Interaction Matrix')
ax.set_xlabel('Products'); ax.set_ylabel('Users')

# Sparsity by interaction type
ax = axes[1]
total_cells = 10000 * 1000
filled = {
    'view':     int((interactions['interaction_type']=='view').sum()),
    'wishlist': int((interactions['interaction_type']=='wishlist').sum()),
    'purchase': int((interactions['interaction_type']=='purchase').sum()),
}
sparsity = {k: 1 - v/total_cells for k,v in filled.items()}
bars = ax.bar(sparsity.keys(),
              [v*100 for v in sparsity.values()],
              color=[BLUE, GOLD, ORANGE], edgecolor='white')
for bar, val in zip(bars, sparsity.values()):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()-0.5,
            f'{val:.2%}', ha='center', va='top',
            fontsize=11, fontweight='bold', color='white')
ax.set_ylim(98, 100.2)
ax.set_title('Matrix Sparsity by Interaction Type')
ax.set_ylabel('Sparsity (%)'); ax.set_ylim(98.5, 100.1)

# Items per user distribution
ax = axes[2]
items_per_user = interactions.groupby('user_id')['product_id'].nunique()
ax.hist(items_per_user.values, bins=50, color=PURPLE, edgecolor='white', linewidth=0.5)
ax.axvline(items_per_user.median(), color=ORANGE, linestyle='--',
           label=f'Median = {items_per_user.median():.0f} items')
ax.axvline(items_per_user.mean(), color=GREEN, linestyle='--',
           label=f'Mean = {items_per_user.mean():.0f} items')
ax.set_title('Unique Items Interacted per User')
ax.set_xlabel('Unique Items'); ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('sparsity_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

overall_sparsity = 1 - len(interactions.groupby(['user_id','product_id']).size()) / total_cells
print(f"Overall matrix sparsity : {overall_sparsity:.2%}")
print(f"Mean items per user     : {items_per_user.mean():.1f}")
print(f"Median items per user   : {items_per_user.median():.1f}")
print(f"Max items for any user  : {items_per_user.max()}")

---
## 7. SHAP Feature Importance Summary

SHAP (SHapley Additive exPlanations) reveals which features drive purchase decisions.
This directly answers **Research Question 2**: *"Which user characteristics are most
predictive in the Nepali e-commerce scenario?"*


In [ ]:
import json, os

shap_path = os.path.join(os.getcwd(), '..', 'data', 'shap_results.json')
with open(shap_path) as f:
    shap_data = json.load(f)

labels = shap_data['feature_labels'][:12]
values = shap_data['shap_values'][:12]

fig, ax = plt.subplots(figsize=(10, 6))
colors_s = [ORANGE if v > 0.1 else BLUE if v > 0.05 else '#CCC' for v in values]
bars = ax.barh(labels[::-1], [v*100 for v in values[::-1]],
               color=colors_s[::-1], edgecolor='white', linewidth=0.5)
ax.set_xlabel('Mean |SHAP value| × 100')
ax.set_title('SHAP Feature Importance — What Drives a Purchase?
'
             f'(Classifier accuracy: {shap_data["classifier_accuracy"]*100:.1f}%)',
             fontsize=13)
ax.axvline(5, color='black', linestyle=':', linewidth=0.8, alpha=0.5)

for bar, val in zip(bars, [v*100 for v in values[::-1]]):
    ax.text(bar.get_width()+0.3, bar.get_y()+bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=10)

plt.tight_layout()
plt.savefig('shap_importance.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nTop 5 most important features:")
for i, (label, val) in enumerate(zip(labels[:5], values[:5]), 1):
    print(f"  {i}. {label:<30} SHAP = {val:.4f}")
print("\nKey finding: Purchase Frequency and RFM Score dominate,")
print("confirming that behavioural history is more predictive")
print("than demographics in the Nepali e-commerce context.")

---
## 8. Key EDA Findings

| Finding | Implication for System Design |
|---------|-------------------------------|
| 95.7% matrix sparsity | Hybrid model essential — CF alone insufficient |
| Festival uplift 1.4–2.8× | Seasonal signals must be built into features |
| Purchase Frequency = top SHAP feature | RFM scoring is a strong user representation |
| 28% remittance receivers | Spending pattern varies — demographic segmentation justified |
| 76% mobile users | UI must be mobile-first |
| Median 44 items per user | Most users have enough history to personalise |
| Kathmandu = 33% of users | City-level geographic weighting is meaningful |
| 84% views, 9% wishlist, 6% purchase | Cold-start users need fallback strategies |

---
*Notebook generated for thesis submission — ST6000CEM Individual Project Preparation*  
*Binnol Dahal · 14809734 · Coventry University / Softwarica College*
